# CLIP-Guided StyleGAN2 — Evaluation & Inference

This notebook adds **model evaluation** and **inference** on top of `3_Mappers.ipynb`.

## What you can do here

| Section | What it does |
|---|---|
| **A – Setup** | Reload checkpoint, models, and helpers |
| **B – Quantitative Evaluation** | CLIP similarity, FID on test set |
| **C – Image → Image Inference** | Give a real face image, generate a GAN face, compare CLIP similarity |
| **D – Text → Image Inference** | Give a text prompt, generate a face, compute text-image CLIP similarity |
| **E – Side-by-Side Visualisation** | Display input vs generated images |


---
## A — Setup
Re-run these cells first (or keep them at the top of your existing notebook).

In [ ]:
# ─── A1  Imports ─────────────────────────────────────────────────────────────
import os, sys, copy, math, torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, utils
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

import clip          # pip install git+https://github.com/openai/CLIP.git

sys.path.append('./stylegan2-ada-pytorch')
import dnnlib, legacy

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

In [ ]:
# ─── A2  Paths — edit these ──────────────────────────────────────────────────
STYLEGAN2_PKL    = '/home/m2s/mind2sketch/Mapping-Neural-Networks/weights/pretrained/stylegan2-ffhq-256x256.pkl'
MAPPER_CKPT      = '/home/m2s/mind2sketch/Mapping-Neural-Networks/weights/secondrun_with_b32/step-XXXXX_lamda-1.4_lr-0.002_batchz-16.pt'  # ← your saved .pt
FFHQ_TEST_DIR    = '/home/m2s/mind2sketch/Mapping-Neural-Networks/ffhq/ffhq256'  # test images
OUTPUT_DIR       = './eval_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ─── A3  Load CLIP ───────────────────────────────────────────────────────────
clip_model, clip_preprocess = clip.load('ViT-B/32', device=device)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False
print('CLIP loaded.')

In [ ]:
# ─── A4  Load StyleGAN2 G & D ────────────────────────────────────────────────
with open(STYLEGAN2_PKL, 'rb') as f:
    model_data = legacy.load_network_pkl(f)

G = model_data['G_ema'].to(device).eval()
D = model_data['D'].to(device).eval()

for p in G.parameters(): p.requires_grad = False
for p in D.parameters(): p.requires_grad = False

num_ws = getattr(G.synthesis, 'num_ws', 14)
w_dim  = getattr(G, 'w_dim', 512)
c_dim  = getattr(G, 'c_dim', 0)
print(f'G loaded — num_ws={num_ws}, w_dim={w_dim}')

In [ ]:
# ─── A5  ThreeMappers class (copy from 3_Mappers.ipynb) ─────────────────────
class ThreeMappers(nn.Module):
    def __init__(self, pretrained_mapping, num_ws=14, w_dim=512, truncation_psi=0.7):
        super().__init__()
        self.num_ws = num_ws
        self.w_dim  = w_dim
        self.truncation_psi = truncation_psi
        self.mappers = nn.ModuleList([copy.deepcopy(pretrained_mapping) for _ in range(3)])
        for mapper in self.mappers:
            for p in mapper.parameters():
                p.requires_grad = True
        self.register_buffer('w_avg', pretrained_mapping.w_avg.clone())
        self.layer_assignments = [
            [0, 1, 2, 3],
            [4, 5, 6, 7],
            list(range(8, num_ws)),
        ]

    def forward(self, clip_embedding):
        w_vectors = []
        for mapper in self.mappers:
            w_broadcast = mapper(clip_embedding, c=None, truncation_psi=1.0)
            w = w_broadcast[:, 0, :]
            w = self.w_avg + self.truncation_psi * (w - self.w_avg)
            w_vectors.append(w)
        layer_to_w = {}
        for mapper_idx, layers in enumerate(self.layer_assignments):
            for layer_idx in layers:
                layer_to_w[layer_idx] = w_vectors[mapper_idx]
        ws = torch.stack([layer_to_w[i] for i in range(self.num_ws)], dim=1)
        return ws

In [ ]:
# ─── A6  Load trained mapper checkpoint ─────────────────────────────────────
mappers = ThreeMappers(G.mapping, num_ws=num_ws, w_dim=w_dim, truncation_psi=0.7).to(device)

ckpt = torch.load(MAPPER_CKPT, map_location=device)
mappers.load_state_dict(ckpt['mappers_state_dict'])
mappers.eval()
print(f"Mapper loaded from step {ckpt.get('step','?')}, epoch {ckpt.get('epoch','?')}")

In [ ]:
# ─── A7  Shared helper functions ─────────────────────────────────────────────
clip_input_size = 224

real_image_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),  # → [-1, 1]
])

@torch.no_grad()
def encode_image_tensor(img_tensor):
    """
    img_tensor: (B, 3, H, W) in [-1, 1]
    Returns:    (B, 512) float32 L2-normalised CLIP embedding
    """
    x = (img_tensor + 1.0) / 2.0
    x = F.interpolate(x, size=(clip_input_size, clip_input_size), mode='bilinear', align_corners=False)
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=x.device).view(1,3,1,1)
    std  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=x.device).view(1,3,1,1)
    x = (x - mean) / std
    emb = clip_model.encode_image(x).float()
    return emb / emb.norm(dim=-1, keepdim=True)

@torch.no_grad()
def encode_text(text: str):
    """
    text: a single string description
    Returns: (1, 512) float32 L2-normalised CLIP text embedding
    """
    tokens = clip.tokenize([text]).to(device)
    emb = clip_model.encode_text(tokens).float()
    return emb / emb.norm(dim=-1, keepdim=True)

@torch.no_grad()
def generate_from_embedding(clip_emb):
    """
    clip_emb: (B, 512)
    Returns:  (B, 3, 256, 256) in [-1, 1]
    """
    ws = mappers(clip_emb)
    return G.synthesis(ws, noise_mode='const')

def load_pil_image(path):
    return Image.open(path).convert('RGB')

def pil_to_tensor(pil_img):
    """PIL → (1, 3, 256, 256) tensor in [-1, 1]"""
    return real_image_transform(pil_img).unsqueeze(0).to(device)

def tensor_to_pil(t):
    """(1,3,H,W) or (3,H,W) tensor in [-1,1] → PIL Image"""
    t = t.squeeze(0).cpu().float()
    t = (t + 1) / 2
    t = t.clamp(0, 1)
    return transforms.ToPILImage()(t)

def show_side_by_side(imgs: list, titles: list, figsize=(14, 5)):
    fig, axes = plt.subplots(1, len(imgs), figsize=figsize)
    if len(imgs) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print('Helpers ready.')

---
## B — Quantitative Evaluation on the Test Set

Computes for every test image:
- **CLIP cosine similarity** between the real image embedding and the generated image embedding  
- Mean ± std reported across the whole test set

> **FID** requires `pytorch-fid`. Install with `pip install pytorch-fid` and uncomment the FID cell.

In [ ]:
# ─── B1  Build test DataLoader ────────────────────────────────────────────────
from torch.utils.data import DataLoader, Dataset

class SimpleImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.image_files = sorted([
            os.path.join(root, f)
            for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

    def __len__(self): return len(self.image_files)

    def __getitem__(self, idx):
        img = Image.open(self.image_files[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

# If you already have a saved test split, re-create it with the same seed=42
from torch.utils.data import random_split

VAL_SPLIT  = 0.1
TEST_SPLIT = 0.1
BATCH_SIZE_EVAL = 16

full_dataset = SimpleImageDataset(FFHQ_TEST_DIR, transform=real_image_transform)
n = len(full_dataset)
val_size  = max(1, int(n * VAL_SPLIT))
test_size = max(1, int(n * TEST_SPLIT))
train_size = n - val_size - test_size

g = torch.Generator().manual_seed(42)   # same seed as training!
train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size], generator=g)

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=2)
print(f'Test set size: {len(test_ds)} images')

In [ ]:
# ─── B2  CLIP similarity evaluation loop ─────────────────────────────────────
from tqdm.auto import tqdm

all_cosine_sims = []
all_adv_scores  = []

mappers.eval()

with torch.no_grad():
    for real_imgs in tqdm(test_loader, desc='Evaluating'):
        real_imgs = real_imgs.to(device)

        # 1) Encode real images with CLIP
        clip_real = encode_image_tensor(real_imgs)         # (B, 512)

        # 2) Generate fake images
        fake_imgs = generate_from_embedding(clip_real)     # (B, 3, 256, 256)

        # 3) Encode fake images with CLIP
        clip_fake = encode_image_tensor(fake_imgs)         # (B, 512)

        # 4) Cosine similarity
        cos_sim = F.cosine_similarity(clip_real, clip_fake, dim=-1)   # (B,)
        all_cosine_sims.extend(cos_sim.cpu().tolist())

        # 5) Discriminator score (higher = more realistic)
        try:
            d_score = D(fake_imgs, c=torch.zeros(real_imgs.shape[0], c_dim, device=device))
        except TypeError:
            d_score = D(fake_imgs)
        if isinstance(d_score, (tuple, list)):
            d_score = d_score[0]
        all_adv_scores.extend(d_score.squeeze().cpu().tolist())

all_cosine_sims = np.array(all_cosine_sims)
all_adv_scores  = np.array(all_adv_scores)

print('─' * 50)
print(f'CLIP Cosine Similarity  →  mean: {all_cosine_sims.mean():.4f}  std: {all_cosine_sims.std():.4f}')
print(f'Discriminator Score     →  mean: {all_adv_scores.mean():.4f}   std: {all_adv_scores.std():.4f}')
print('─' * 50)

In [ ]:
# ─── B3  Plot CLIP similarity distribution ───────────────────────────────────
plt.figure(figsize=(8, 4))
plt.hist(all_cosine_sims, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
plt.axvline(all_cosine_sims.mean(), color='red', linestyle='--', label=f'Mean = {all_cosine_sims.mean():.3f}')
plt.xlabel('CLIP Cosine Similarity  (real vs generated)')
plt.ylabel('Count')
plt.title('Test-Set CLIP Similarity Distribution')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'clip_sim_distribution.png'), dpi=150)
plt.show()

In [ ]:
# ─── B4  (Optional) FID score ─────────────────────────────────────────────────
# pip install pytorch-fid
# Uncomment and run only if pytorch-fid is installed.

# import shutil
# from pytorch_fid import fid_score
#
# REAL_DIR = os.path.join(OUTPUT_DIR, 'fid_real')
# FAKE_DIR = os.path.join(OUTPUT_DIR, 'fid_fake')
# os.makedirs(REAL_DIR, exist_ok=True)
# os.makedirs(FAKE_DIR, exist_ok=True)
#
# with torch.no_grad():
#     for i, real_imgs in enumerate(tqdm(test_loader, desc='Saving FID images')):
#         real_imgs = real_imgs.to(device)
#         clip_real = encode_image_tensor(real_imgs)
#         fake_imgs = generate_from_embedding(clip_real)
#         for j in range(real_imgs.shape[0]):
#             idx = i * BATCH_SIZE_EVAL + j
#             tensor_to_pil(real_imgs[j]).save(os.path.join(REAL_DIR, f'{idx:05d}.png'))
#             tensor_to_pil(fake_imgs[j]).save(os.path.join(FAKE_DIR, f'{idx:05d}.png'))
#
# fid_value = fid_score.calculate_fid_given_paths(
#     [REAL_DIR, FAKE_DIR], batch_size=32, device=device, dims=2048
# )
# print(f'FID = {fid_value:.2f}')

---
## C — Image → Image Inference

Give **any face image** (file path or PIL image).  
The pipeline encodes it with CLIP, passes the embedding through the trained mappers,  
generates a face with the frozen StyleGAN2 synthesis network, then measures similarity.

In [ ]:
# ─── C1  Single image inference ──────────────────────────────────────────────

IMAGE_PATH = '/path/to/your/face_image.jpg'   # ← change this

# Load & preprocess
pil_input   = load_pil_image(IMAGE_PATH)
input_tensor = pil_to_tensor(pil_input)        # (1, 3, 256, 256) in [-1,1]

with torch.no_grad():
    # 1) CLIP encode the input image
    clip_emb = encode_image_tensor(input_tensor)   # (1, 512)

    # 2) Generate via mappers + synthesis
    fake_tensor = generate_from_embedding(clip_emb) # (1, 3, 256, 256)

    # 3) CLIP encode the generated image
    clip_fake = encode_image_tensor(fake_tensor)    # (1, 512)

    # 4) Similarity
    sim = F.cosine_similarity(clip_emb, clip_fake, dim=-1).item()

pil_generated = tensor_to_pil(fake_tensor)

print(f'CLIP cosine similarity (input image vs generated image): {sim:.4f}')

show_side_by_side(
    [pil_input.resize((256,256)), pil_generated],
    [f'Input image', f'Generated (CLIP sim = {sim:.3f})']
)

In [ ]:
# ─── C2  Batch inference on multiple images ────────────────────────────────
# Provide a list of paths; results are saved to OUTPUT_DIR

IMAGE_PATHS = [
    '/path/to/image1.jpg',   # ← change
    '/path/to/image2.jpg',
]

results = []
for path in IMAGE_PATHS:
    pil_img = load_pil_image(path)
    t = pil_to_tensor(pil_img)

    with torch.no_grad():
        emb  = encode_image_tensor(t)
        fake = generate_from_embedding(emb)
        emb_fake = encode_image_tensor(fake)
        sim = F.cosine_similarity(emb, emb_fake, dim=-1).item()

    results.append({'path': path, 'sim': sim, 'generated': tensor_to_pil(fake)})
    print(f'{os.path.basename(path)}  →  CLIP sim = {sim:.4f}')

# Display grid
all_imgs   = [load_pil_image(r['path']).resize((256,256)) for r in results] + \
             [r['generated'] for r in results]
all_titles = [f"Input: {os.path.basename(r['path'])}" for r in results] + \
             [f"Generated (sim={r['sim']:.3f})" for r in results]

n = len(results)
fig, axes = plt.subplots(2, n, figsize=(4*n, 9))
for col, (img, title) in enumerate(zip(all_imgs, all_titles)):
    row = 0 if col < n else 1
    c   = col % n
    axes[row][c].imshow(img)
    axes[row][c].set_title(title, fontsize=9)
    axes[row][c].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'batch_image_inference.png'), dpi=150)
plt.show()

---
## D — Text → Image Inference

Give a **text description** (e.g. `"a smiling young woman with curly hair"`).

CLIP encodes the text into a 512-dim embedding — **the same embedding space used for images** — then passes it through the trained mappers to generate a face.  
We measure **text-to-image CLIP similarity** to quantify how well the generation reflects the description.

In [ ]:
# ─── D1  Text → Image ─────────────────────────────────────────────────────────

TEXT_PROMPT = 'a smiling young woman with curly blonde hair'   # ← change this

with torch.no_grad():
    # 1) Encode text with CLIP
    text_emb = encode_text(TEXT_PROMPT)          # (1, 512)

    # 2) Generate via mappers + synthesis
    fake_tensor = generate_from_embedding(text_emb)  # (1, 3, 256, 256)

    # 3) Encode generated image with CLIP
    img_emb = encode_image_tensor(fake_tensor)    # (1, 512)

    # 4) Text-Image similarity
    text_img_sim = F.cosine_similarity(text_emb, img_emb, dim=-1).item()

pil_out = tensor_to_pil(fake_tensor)

print(f'Prompt: "{TEXT_PROMPT}"')
print(f'Text-Image CLIP similarity: {text_img_sim:.4f}')

plt.figure(figsize=(4, 4))
plt.imshow(pil_out)
plt.title(f'"{TEXT_PROMPT}"\nCLIP sim = {text_img_sim:.3f}', fontsize=9)
plt.axis('off')
plt.tight_layout()
save_name = TEXT_PROMPT[:40].replace(' ', '_') + '.png'
plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
plt.show()

In [ ]:
# ─── D2  Compare multiple text prompts side by side ──────────────────────────

PROMPTS = [
    'a young man with short dark hair',
    'an elderly woman with glasses',
    'a smiling child with freckles',
    'a bearded man wearing a hat',
]

gen_images = []
sims = []

with torch.no_grad():
    for prompt in PROMPTS:
        t_emb  = encode_text(prompt)
        fake   = generate_from_embedding(t_emb)
        i_emb  = encode_image_tensor(fake)
        sim    = F.cosine_similarity(t_emb, i_emb, dim=-1).item()
        gen_images.append(tensor_to_pil(fake))
        sims.append(sim)
        print(f'{prompt!r}  →  sim = {sim:.4f}')

n = len(PROMPTS)
fig, axes = plt.subplots(1, n, figsize=(4*n, 5))
for ax, img, prompt, sim in zip(axes, gen_images, PROMPTS, sims):
    ax.imshow(img)
    ax.set_title(f'"{prompt}"\nsim={sim:.3f}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'text_prompts_grid.png'), dpi=150)
plt.show()

---
## E — Cross-Modal Similarity: Image Input vs Text Description

Here you can give **both** a reference image **and** a text description, then check how similar the generated image is to each.

In [ ]:
# ─── E1  Image + Text → dual similarity ─────────────────────────────────────

IMAGE_PATH  = '/path/to/your/face.jpg'            # ← change
TEXT_PROMPT = 'a person with dark hair smiling'   # ← change (describe the image)

pil_input    = load_pil_image(IMAGE_PATH)
input_tensor = pil_to_tensor(pil_input)

with torch.no_grad():
    # Embeddings
    img_emb_real = encode_image_tensor(input_tensor)  # (1,512) — from real image
    txt_emb      = encode_text(TEXT_PROMPT)           # (1,512) — from text

    # Two generation paths
    fake_from_image = generate_from_embedding(img_emb_real)  # conditioned on image
    fake_from_text  = generate_from_embedding(txt_emb)       # conditioned on text

    # Encode the generated images
    emb_gen_img = encode_image_tensor(fake_from_image)
    emb_gen_txt = encode_image_tensor(fake_from_text)

    # Similarities
    sim_img2gen = F.cosine_similarity(img_emb_real, emb_gen_img, dim=-1).item()
    sim_txt2gen = F.cosine_similarity(txt_emb,      emb_gen_txt, dim=-1).item()
    sim_img_txt = F.cosine_similarity(img_emb_real, txt_emb,     dim=-1).item()

print(f'Image → Generated   CLIP sim: {sim_img2gen:.4f}')
print(f'Text  → Generated   CLIP sim: {sim_txt2gen:.4f}')
print(f'Image ↔ Text        CLIP sim: {sim_img_txt:.4f}  (how well the text describes the image)')

show_side_by_side(
    [pil_input.resize((256,256)), tensor_to_pil(fake_from_image), tensor_to_pil(fake_from_text)],
    [
        'Input Image',
        f'Generated from Image\n(sim={sim_img2gen:.3f})',
        f'Generated from Text\n"{TEXT_PROMPT}"\n(sim={sim_txt2gen:.3f})'
    ],
    figsize=(14, 5)
)

---
## Summary of Metrics

| Metric | Description | Target |
|---|---|---|
| **CLIP cosine sim (image→gen)** | How semantically similar the generated face is to the input | Higher is better (>0.25 is reasonable) |
| **CLIP cosine sim (text→gen)** | How well the generated face matches the text description | Higher is better |
| **Discriminator score** | How realistic the generated image looks | Higher is better (>0) |
| **FID** *(optional)* | Distribution distance between real and generated sets | Lower is better (<50 is good) |
